# Interview Data Collection

This notebook collects tennis interview pages and extracts pre-match interview content for the 2020-2026 period.

The scraping pipeline stores the interview metadata, source URLs, and extracted question-answer pairs. The resulting files are later matched with the cleaned tournament data in the dataset-construction notebook.

> **Execution note:** This notebook was designed to run in Google Colab and uses Google Drive for persistent storage. Collecting the full interview dataset takes more than 10 hours, so intermediate results and progress checkpoints are saved to Drive throughout the scraping process. This allows the collection process to resume after interruptions without restarting from the beginning.
>
> The notebook expects a `tennis_project` directory under `MyDrive` and uses `/content/drive/MyDrive/tennis_project` as its output directory. The directory is created automatically if it does not already exist.
>
> The datasets produced by the completed scraping run are included with the project submission, so rerunning the full interview collection process is not required.

## Google Colab Environment Setup

The following commands install Chrome and the Python packages required by the Selenium-based interview scraper.

In [ ]:
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get update -qq
!apt-get install -y ./google-chrome-stable_current_amd64.deb
!pip install -U selenium
!pip install -U selenium beautifulsoup4

In [ ]:
import csv
import re
import string
import time
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import json
from datetime import datetime
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
import os
import shutil
from google.colab import drive

In [ ]:
drive.mount("/content/drive")

OUTPUT_DIR = "/content/drive/MyDrive/tennis_project"
os.makedirs(OUTPUT_DIR, exist_ok=True)


LOCAL_PLAYERS_CSV_PATH = "tennis_players_with_interviews_2020_2026.csv"
OUTPUT_CSV_PATH = f"{OUTPUT_DIR}/pre_match_interviews_2020_2026.csv"
STATUS_OUTPUT_CSV_PATH = f"{OUTPUT_DIR}/tournament_interview_status_2020_2026.csv"
COMPLETED_PLAYERS_PATH = f"{OUTPUT_DIR}/completed_players_2020_2026.txt"
PLAYERS_CSV_PATH = f"{OUTPUT_DIR}/{LOCAL_PLAYERS_CSV_PATH}"

START_YEAR = 2020
END_YEAR = 2026

## Player Page Collection

We first collect the ASAP Sports tennis player pages and retain players with at least one interview during 2020-2026. The resulting player names and URLs are then used as input to the interview scraper.

In [ ]:
# --------------------------------------------------
# Settings
# --------------------------------------------------

BASE_URL = "https://www.asapsports.com"
LETTER_URL = "https://www.asapsports.com/show_player.php?category=7&letter={letter}"

REQUEST_DELAY_SECONDS = 0.2
REQUEST_TIMEOUT_SECONDS = 30

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (X11; Linux x86_64) "
        "AppleWebKit/537.36 "
        "Chrome/120.0 Safari/537.36"
    )
}


# --------------------------------------------------
# Helper functions
# --------------------------------------------------

def clean_text(value):
    return re.sub(r"\s+", " ", value).strip()


def convert_player_name(name):
    """
    convert:
    'Badosa, Paula' -> 'Paula Badosa'
    """
    name = clean_text(name)

    if "," not in name:
        return name

    last_name, first_name = name.split(",", 1)
    return clean_text(f"{first_name} {last_name}")


def extract_players_from_letter(session, letter):
    """
    Collects all players from a page of a particular letter.
    """
    url = LETTER_URL.format(letter=letter)

    response = session.get(
        url,
        headers=HEADERS,
        timeout=REQUEST_TIMEOUT_SECONDS,
    )

    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    players = {}

    for link in soup.find_all("a", href=True):
        href = link["href"]

        if not re.search(r"show_player\.php\?id=\d+", href):
            continue

        original_name = clean_text(link.get_text(" ", strip=True))

        if not original_name:
            continue

        player_name = convert_player_name(original_name)
        player_url = urljoin(BASE_URL, href)
        players[player_name] = player_url

    return players


def get_all_tennis_players(session):
    """
    Goes through all the letters and returns all the tennis players.
    """
    all_players = {}

    for letter in string.ascii_lowercase:
        try:
            letter_players = extract_players_from_letter(session, letter)
            all_players.update(letter_players)

            print(
                f"Letter {letter.upper()}: "
                f"found {len(letter_players)} players"
            )

        except requests.RequestException as error:
            print(f"Letter {letter.upper()}: failed - {error}")

        time.sleep(REQUEST_DELAY_SECONDS)

    return dict(sorted(all_players.items()))


def has_interview_between_years(session, player_url, start_year, end_year):
    """
    Checks if the player has at least one interview from start_year to end_year.
    """
    response = session.get(
        player_url,
        headers=HEADERS,
        timeout=REQUEST_TIMEOUT_SECONDS,
    )

    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    for row in soup.find_all("tr"):
        row_text = row.get_text(" ", strip=True)
        year_matches = re.findall(r"\b(?:19|20)\d{2}\b", row_text)

        for year_text in year_matches:
            year = int(year_text)

            if start_year <= year <= end_year:
                interview_link = row.find(
                    "a",
                    href=lambda href: (
                        href
                        and "show_interview.php" in href
                    ),
                )

                if interview_link:
                    return True

    return False


def filter_players_by_year(
    session,
    players,
    start_year,
    end_year,
):
    """
    Only keeps players who have an interview between the years start_year-end_year.
    """
    filtered_players = {}
    failed_players = {}
    total = len(players)

    for index, (player_name, player_url) in enumerate(players.items(), start=1):
        print(f"{index}/{total} - checking {player_name}")

        try:
            if has_interview_between_years(
                session,
                player_url,
                start_year,
                end_year,
            ):
                filtered_players[player_name] = player_url
                print("  Included")
            else:
                print("  No matching interview")

        except requests.RequestException as error:
            failed_players[player_name] = {
                "url": player_url,
                "error": str(error),
            }

            print(f"  Failed: {error}")

        time.sleep(REQUEST_DELAY_SECONDS)

    return filtered_players, failed_players


def save_players_as_csv(players, output_path):
    """
    Saves the names and links in a CSV file.
    """
    with open(output_path, "w", newline="", encoding="utf-8-sig") as file:
        writer = csv.DictWriter(file, fieldnames=["player", "url"])
        writer.writeheader()

        for player_name, player_url in players.items():
            writer.writerow({"player": player_name, "url": player_url})


def print_players_summary(players, n_examples=10):
    print(f"Players with interviews found: {len(players)}")
    print(f"\nFirst {min(n_examples, len(players))} players:")

    for player_name, player_url in list(sorted(players.items()))[:n_examples]:
        print(f"  {player_name}: {player_url}")

# --------------------------------------------------
# Run
# --------------------------------------------------

def get_players_names_and_links():
    with requests.Session() as session:
        print("Collecting all tennis players...\n")

        all_players = get_all_tennis_players(session)

        print(f"\nTotal tennis players found: {len(all_players)}")

        print(f"\nFiltering players with interviews from {START_YEAR} to {END_YEAR}...\n")

        filtered_players, failed_players = (
            filter_players_by_year(
                session,
                all_players,
                START_YEAR,
                END_YEAR,
            )
        )

    players_names_and_links = dict(sorted(filtered_players.items()))
    print("Removing test players ('player test', 'Player Testing', 'Speaker Testing')")
    players_names_and_links.pop("player test", None)
    players_names_and_links.pop("Player Testing", None)
    players_names_and_links.pop("Speaker Testing", None)

    print("\nSummary")
    print(f"All players: {len(all_players)}")
    print(f"Players with interviews from {START_YEAR} to {END_YEAR}: {len(players_names_and_links)}")
    print(f"Failed player pages: {len(failed_players)}")

    save_players_as_csv(players_names_and_links, LOCAL_PLAYERS_CSV_PATH)
    print(f"Saved CSV file to: {LOCAL_PLAYERS_CSV_PATH}")
    print_players_summary(players_names_and_links)
    return players_names_and_links

get_players_names_and_links()

## Interview Collection

For each selected player, we collect interview pages from 2020-2026, identify pre-match interviews, and extract interview metadata and question-answer content.

Intermediate results and player-level completion checkpoints are saved to Google Drive throughout the process, allowing the scraper to resume after interruptions.

In [ ]:
REQUEST_DELAY_SECONDS = 0.01
MAX_INTERVIEWS_PER_PLAYER = None


# On the first run, copy the locally generated player file to Drive
# if a persistent copy does not exist yet.
if not os.path.exists(PLAYERS_CSV_PATH):
    if os.path.exists(LOCAL_PLAYERS_CSV_PATH):
        shutil.copy(LOCAL_PLAYERS_CSV_PATH, PLAYERS_CSV_PATH)
    else:
        raise FileNotFoundError(f"Players file not found: {PLAYERS_CSV_PATH}")

# Load player names and URLs
players_names_and_links_df = pd.read_csv(PLAYERS_CSV_PATH)
PLAYERS = dict(zip(players_names_and_links_df["player"], players_names_and_links_df["url"]))

BASE_URL = "https://www.asapsports.com"


def create_driver():
    options = Options()
    options.binary_location = "/usr/bin/google-chrome"
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--lang=en-US")
    options.add_argument("--remote-debugging-port=9222")

    driver = webdriver.Chrome(options=options)
    driver.set_page_load_timeout(30)

    return driver


def fetch_soup(driver, url):
    time.sleep(REQUEST_DELAY_SECONDS)
    driver.get(url)

    WebDriverWait(driver, 20).until(
        lambda d: d.execute_script("return document.readyState") == "complete"
    )

    return BeautifulSoup(driver.page_source, "html.parser")


def clean_text(value):
    return re.sub(r"\s+", " ", value).strip()


def get_all_lines(soup):
    text = soup.get_text("\n")
    return [line.strip() for line in text.splitlines() if line.strip()]


def parse_date_line(line):
    for fmt in ["%B %d, %Y", "%b %d, %Y"]:
        try:
            return datetime.strptime(line.strip(), fmt).date().isoformat()
        except ValueError:
            pass

    return None


def extract_header_metadata(soup):
    lines = get_all_lines(soup)

    for i in range(len(lines) - 2):
        interview_date = parse_date_line(lines[i + 1])

        if interview_date:
            return {
                "tournament": clean_text(lines[i]),
                "interview_date": interview_date,
                "player_from_page": clean_text(lines[i + 2]),
                "content_start_index": i + 3,
            }

    return {
        "tournament": None,
        "interview_date": None,
        "player_from_page": None,
        "content_start_index": 0,
    }


def extract_interview_links(driver, player_name, player_url, start_year, end_year):
    soup = fetch_soup(driver, player_url)
    results = []

    for row in soup.find_all("tr"):
        row_text = row.get_text(" ", strip=True)

        year_match = re.search(r"\b(19|20)\d{2}\b", row_text)
        if not year_match:
            continue

        year = int(year_match.group(0))

        if not (start_year <= year <= end_year):
            continue

        link = row.find("a", href=lambda h: h and "show_interview.php" in h)
        if not link:
            continue

        results.append({
            "player": player_name,
            "year": year,
            "event": link.get_text(" ", strip=True),
            "url": urljoin(BASE_URL, link["href"]),
        })

    return results


def looks_like_score_line(line):
    return bool(re.search(r"\b\d{1,2}-\d{1,2}\b", line))


def is_question_start(line):
    line = line.strip()
    return line.startswith("Q.") or line.startswith("THE MODERATOR:")


def is_real_question_start(line):
    if should_ignore_moderator_line(line):
        return False

    return is_question_start(line)


def is_player_answer_start(line, player_name):
    player_upper = player_name.upper()
    line_upper = line.strip().upper()
    return line_upper.startswith(player_upper + ":")


def should_ignore_moderator_line(line):
    normalized = clean_text(line).lower()
    return normalized.startswith("the moderator: questions")


def is_pre_match_interview_soup(soup, player_name, metadata):
    lines = get_all_lines(soup)
    start_index = metadata.get("content_start_index") or 0

    for i in range(start_index, len(lines)):
        line = lines[i]

        if "FastScripts Transcript by ASAP Sports" in line:
            return True

        if is_real_question_start(line) or is_player_answer_start(line, player_name):
            return True

        if looks_like_score_line(line):
            return False

    return True


def remove_question_prefix(line):
    line = line.strip()

    if line.startswith("Q."):
        return re.sub(r"^Q\.\s*", "", line).strip()

    if line.startswith("THE MODERATOR:"):
        return re.sub(r"^THE MODERATOR:\s*", "", line).strip()

    return line


def remove_answer_prefix(line, player_name):
    return re.sub(
        rf"^{re.escape(player_name.upper())}:\s*",
        "",
        line.strip(),
        flags=re.IGNORECASE,
    ).strip()


def extract_questions_and_answers(soup, player_name, metadata):
    lines = get_all_lines(soup)

    start_index = metadata.get("content_start_index") or 0
    end_index = len(lines)

    for i in range(start_index, len(lines)):
        if lines[i].lower() == "press conference":
            start_index = i + 1
            break

    for i in range(start_index, len(lines)):
        if "FastScripts Transcript by ASAP Sports" in lines[i]:
            end_index = i
            break

    content_lines = lines[start_index:end_index]

    qa = {}
    question_counter = 0

    current_question_lines = []
    current_answer_lines = []
    state = "searching_question"

    for line in content_lines:
        if should_ignore_moderator_line(line):
            continue

        if looks_like_score_line(line):
            continue

        if state == "searching_question":
            if is_question_start(line):
                question_text = remove_question_prefix(line)
                current_question_lines = [question_text] if question_text else []
                current_answer_lines = []
                state = "reading_question"
            continue

        if state == "reading_question":
            if is_player_answer_start(line, player_name):
                current_answer_lines = [remove_answer_prefix(line, player_name)]
                state = "reading_answer"
            else:
                current_question_lines.append(line)
            continue

        if state == "reading_answer":
            if is_question_start(line):
                question_counter += 1
                qa[f"question_{question_counter}"] = clean_text(
                    " ".join(current_question_lines)
                )
                qa[f"answer_{question_counter}"] = clean_text(
                    " ".join(current_answer_lines)
                )

                question_text = remove_question_prefix(line)
                current_question_lines = [question_text] if question_text else []
                current_answer_lines = []
                state = "reading_question"
            else:
                current_answer_lines.append(line)

    if current_question_lines and current_answer_lines:
        question_counter += 1
        qa[f"question_{question_counter}"] = clean_text(
            " ".join(current_question_lines)
        )
        qa[f"answer_{question_counter}"] = clean_text(
            " ".join(current_answer_lines)
        )

    return qa


def write_rows_to_csv(rows, output_csv_path):
    columns = [
        "player",
        "interview_date",
        "tournament",
        "tournament_finish_position",
        "url",
        "qa_json",
    ]

    temp_path = output_csv_path + ".tmp"

    with open(temp_path, "w", newline="", encoding="utf-8-sig") as file:
        writer = csv.DictWriter(file, fieldnames=columns)
        writer.writeheader()
        writer.writerows(rows)

    os.replace(temp_path, output_csv_path)


def write_status_rows_to_csv(rows, output_csv_path):
    columns = [
        "player",
        "year",
        "tournament",
        "has_any_interview",
        "has_pre_match_interview",
    ]

    temp_path = output_csv_path + ".tmp"

    with open(temp_path, "w", newline="", encoding="utf-8-sig") as file:
        writer = csv.DictWriter(file, fieldnames=columns)
        writer.writeheader()
        writer.writerows(rows)

    os.replace(temp_path, output_csv_path)


def load_existing_rows(file_path):
    if not os.path.exists(file_path):
        return []

    with open(file_path, "r", encoding="utf-8-sig") as file:
        return list(csv.DictReader(file))


def load_existing_status(file_path):
    if not os.path.exists(file_path):
        return {}

    tournament_status = {}

    with open(file_path, "r", encoding="utf-8-sig") as file:
        reader = csv.DictReader(file)

        for row in reader:
            year = int(row["year"])

            row["year"] = year
            row["has_any_interview"] = row["has_any_interview"].lower() == "true"
            row["has_pre_match_interview"] = row["has_pre_match_interview"].lower() == "true"

            status_key = (row["player"], year, row["tournament"])
            tournament_status[status_key] = row

    return tournament_status


def load_completed_players(file_path):
    if not os.path.exists(file_path):
        return set()

    with open(file_path, "r", encoding="utf-8") as file:
        return {line.strip() for line in file if line.strip()}


def mark_player_completed(player_name, file_path):
    with open(file_path, "a", encoding="utf-8") as file:
        file.write(player_name + "\n")


def main():
    driver = None

    # Load the previous checkpoint, if available.
    rows = load_existing_rows(OUTPUT_CSV_PATH)
    tournament_status = load_existing_status(STATUS_OUTPUT_CSV_PATH)
    completed_players = load_completed_players(COMPLETED_PLAYERS_PATH)

    total_players = len(PLAYERS)

    print(f"Already completed: {len(completed_players)}/{total_players} players")
    print(f"Loaded {len(rows)} existing pre-match interviews")
    print(f"Loaded {len(tournament_status)} existing tournament statuses")

    try:
        driver = create_driver()

        for player_num, (player_name, player_url) in enumerate(PLAYERS.items(), start=1):

            if player_name in completed_players:
                print(f"\nSkipping player {player_num}/{total_players}: {player_name}")
                continue

            print(f"\nProcessing player {player_num}/{total_players}: {player_name}")

            # If a previous run stopped after saving the CSV files but before
            # marking the player as completed, remove the player's existing
            # records and process the player again.
            rows = [row for row in rows if row["player"] != player_name]

            player_status_keys = [
                key for key in tournament_status
                if key[0] == player_name
            ]

            for key in player_status_keys:
                del tournament_status[key]

            processed_urls = set()
            completed_tournaments = set()

            links = extract_interview_links(driver, player_name, player_url, START_YEAR, END_YEAR)

            if MAX_INTERVIEWS_PER_PLAYER is not None:
                links = links[:MAX_INTERVIEWS_PER_PLAYER]

            for link_num, item in enumerate(links):
                print(f"  Processing link {link_num + 1}/{len(links)} ...")

                # Used internally for efficient skipping. This key is not stored in the output files.
                event_key = (item["year"], item["event"])

                # A pre-match interview was already found for this tournament.
                if event_key in completed_tournaments:
                    continue

                # Skip interviews that were already processed successfully.
                if item["url"] in processed_urls:
                    continue

                try:
                    soup = fetch_soup(driver, item["url"])
                    metadata = extract_header_metadata(soup)

                    tournament_name = metadata["tournament"]

                    if not tournament_name:
                        print(f"  Tournament name not found: {item['url']}")
                        continue

                    # Mark the interview as processed only after the page
                    # has been loaded and parsed successfully.
                    processed_urls.add(item["url"])

                    status_key = (item["player"], item["year"], tournament_name)

                    # Finding a valid interview page confirms that the player
                    # had at least one interview for this tournament.
                    if status_key not in tournament_status:
                        tournament_status[status_key] = {
                            "player": item["player"],
                            "year": item["year"],
                            "tournament": tournament_name,
                            "has_any_interview": True,
                            "has_pre_match_interview": False,
                        }

                    # If this is not a pre-match interview, continue in the
                    # original order in which interviews appear on ASAP Sports.
                    if not is_pre_match_interview_soup(soup, item["player"], metadata):
                        continue

                    qa = extract_questions_and_answers(soup, item["player"], metadata)

                    if not qa:
                        continue

                    # Keep the first pre-match interview encountered in the
                    # original ASAP Sports ordering for this tournament.
                    tournament_status[status_key]["has_pre_match_interview"] = True
                    completed_tournaments.add(event_key)


                    # Placeholder for tournament_finish_position. The actual tournament outcome
                    # is added later when the interview data is merged with the tournament dataset.
                    row = {
                        "player": item["player"],
                        "interview_date": metadata["interview_date"],
                        "tournament": tournament_name,
                        "tournament_finish_position": 0,
                        "url": item["url"],
                        "qa_json": json.dumps(qa, ensure_ascii=False),
                    }

                    rows.append(row)

                    print(
                        f'  {row["player"]} | {row["interview_date"]} | '
                        f'{row["tournament"]} | finish={row["tournament_finish_position"]} | '
                        f'{row["url"]}'
                    )

                except Exception as exp:
                    print(f"  Got Error processing {item['url']}: {exp}")

            # --------------------------------------------------
            # CHECKPOINT - save after processing the entire player
            # --------------------------------------------------

            status_rows = list(tournament_status.values())

            write_rows_to_csv(rows, OUTPUT_CSV_PATH)
            write_status_rows_to_csv(status_rows, STATUS_OUTPUT_CSV_PATH)

            # Mark the player as completed only after both files have been saved successfully.
            mark_player_completed(player_name, COMPLETED_PLAYERS_PATH)
            completed_players.add(player_name)

            print(f"  Checkpoint saved - completed {len(completed_players)}/{total_players} players")

    except KeyboardInterrupt:
        print("\nStopped by user.")

    finally:
        if driver is not None:
            try:
                driver.quit()
            except Exception:
                pass

        # Save the current progress even if the run is manually
        # interrupted while processing a player.
        status_rows = list(tournament_status.values())

        write_rows_to_csv(rows, OUTPUT_CSV_PATH)
        write_status_rows_to_csv(status_rows, STATUS_OUTPUT_CSV_PATH)

        print(f"\nSaved {len(rows)} pre-match interviews to {OUTPUT_CSV_PATH}")
        print(f"Saved {len(status_rows)} tournament statuses to {STATUS_OUTPUT_CSV_PATH}")
        print(f"Completed {len(completed_players)}/{total_players} players")


if __name__ == "__main__":
    main()